# Load your replicates into MDAnalysis

This notebook loads one replicate of your project as an MDAnalysis `Universe`,
in two ways:

- with PolyzyMD's loader, from the study's configs: the universe, frames and
  times that `polyzymd analyze` and your own `function:` analyses use;
- by hand, from the files the config points to, for MDAnalysis code or tools
  of your own.

Then it selects atoms and measures one quantity over the frames. Change any
cell and run it again. Nothing here writes into your project.

`polyzymd project init` puts a copy of this notebook in your project's
`notebooks/` folder. Open it there in the PolyzyMD `build` environment, which
has JupyterLab; the docs page "Analyze the trajectories with your own tools"
gives the commands, the files of a replicate and the names in a built system.

## 1. Choose a replicate

By default: the first study in `project.yaml`, its control (its first
condition), and the first replicate with production frames. Set a label or a
replicate number to choose another. To load a config outside a project, set
`config_path`.

In [ ]:
from pathlib import Path

import polyzymd as pz
from polyzymd.analyses.exceptions import ProtocolError

project_folder = Path("..")  # this notebook sits in <project>/notebooks
study_label = None  # None: the first study in project.yaml
condition_label = None  # None: the study's control, its first condition
replicate_index = None  # None: the first replicate with production frames
config_path = None  # or the path of any config.yaml, to load it without a project
equilibration = "0ns"  # the window for config_path; a study uses its own

if config_path is None:
    project = pz.Project(project_folder)
    print(project)
    try:
        study = project[study_label or project.labels[0]]
    except ProtocolError as error:
        raise ProtocolError(
            f"{error}\nA study fresh from project init lists no conditions yet: add one with "
            "polyzymd study add-condition, build and run it, then run this cell again."
        ) from None
else:
    study = pz.Study.from_configs([config_path], equilibration=equilibration)
for condition in study.conditions:
    print(f"{condition.label}: {condition.config_path}")
    print(f"  replicates with production frames: {[r.index for r in condition.replicates]}")
    if condition.not_run:
        print(f"  not run yet: {condition.not_run}")

condition = study[condition_label or study.control]
assert condition.replicates, f"{condition.label!r} has no replicate with production frames yet."
if replicate_index is None:
    replicate = condition.replicates[0]
else:
    replicate = next((r for r in condition.replicates if r.index == replicate_index), None)
    assert replicate is not None, f"{condition.label!r} has no replicate {replicate_index} with frames."
print("Chosen:", replicate)

## 2. PolyzyMD's universe

`replicate.universe()` joins the production segments in order. The atoms carry
the chain IDs, segment IDs and elements of `solvated_system.pdb`; GROMACS runs
are read from their `.top` file when MDAnalysis cannot read `prod.tpr`.

The universe holds every production frame. `replicate.frames` gives the frames
PolyzyMD analyzes: those after the study's equilibration window (and its
`stride`), without a frame that two segments both wrote. `replicate.times`
gives their times in ns. The trajectory itself stores ps.

In [ ]:
u = replicate.universe()
frames, times = replicate.frames, replicate.times
print(u)
print(
    f"{u.trajectory.n_frames} production frames; {len(frames)} after the "
    f"{condition.equilibration} window, from {times[0]:.3f} to {times[-1]:.3f} ns"
)
for segment in u.segments:
    if len(segment.atoms) == 0:  # the topology's own segments, emptied by PolyzyMD's IDs
        continue
    print(f"segment {segment.segid:5s} {len(segment.atoms):7d} atoms {len(segment.residues):6d} residues")

print("\nResidue names and the atom names of their first residue:")
for resname in sorted(set(u.residues.resnames)):
    names = u.select_atoms(f"resname {resname}").residues[0].atoms.names
    shown = " ".join(names[:14]) + (" ..." if len(names) > 14 else "")
    print(f"  {resname:5s} {shown}")

## 3. Select atoms

The built systems name water `HOH` with oxygen `O1x`, and ions `Na+` and
`Cl-`, so selections such as `resname SOL` or `name OW` match nothing. These
selections work in every PolyzyMD system; a component the system does not
have selects 0 atoms.

In [ ]:
selections = {
    "protein": "protein",
    "protein heavy atoms": "protein and not element H",
    "polymer": "segid POLY",
    "ligand": "segid LIG",
    "water oxygens": "resname HOH and name O1x",
    "ions": "segid ION",
    "water and co-solvents": "segid SOL",
}
for what, selection in selections.items():
    print(f"{what:22s} {selection!r:32s} {len(u.select_atoms(selection)):7d} atoms")

## 4. Measure over the frames

The radius of gyration of the protein in each analyzed frame. Replace the
function with your own measurement: it gets the atoms of the current frame.

In [ ]:
import numpy as np

atoms = u.select_atoms("protein")
if len(atoms) == 0:  # a system without a protein: measure every solute atom
    atoms = u.select_atoms("not segid SOL ION")


def measure(atoms):
    return atoms.radius_of_gyration()  # in A


values = np.array([measure(atoms) for _ in u.trajectory[frames]])
print(f"radius of gyration: mean {values.mean():.2f} A, standard deviation {values.std():.2f} A, "
      f"over {len(values)} frames")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
if plt is not None:
    figure, axes = plt.subplots(figsize=(7, 3))
    axes.plot(times, values)
    axes.set_xlabel("production time (ns)")
    axes.set_ylabel("radius of gyration (A)")
    axes.set_title(f"{condition.label}, replicate {replicate.index}")
    plt.show()

## 5. The same replicate by hand

What the loader did, from the files of the replicate folder, for code or tools
of your own. OpenMM runs: `system.prmtop` (or `solvated_system.pdb`) with the
production DCDs, sorted by segment number. GROMACS runs: the `.top` file with
`prod_centered.xtc`. The topology has no chain IDs and its segments are not
PolyzyMD's, so they are copied from `solvated_system.pdb`.

This route does not skip the equilibration window, and does not remove a frame
that two segments both wrote; select the frames yourself.

In [ ]:
import re

import MDAnalysis as mda

folders = dict(condition.config.discover_replicate_dirs())
run_dir = Path(folders[replicate.index]).resolve()
print("Replicate folder:", run_dir)


def add_build_ids(universe, pdb_file):
    """Copy chain IDs, segment IDs and, where missing, elements from solvated_system.pdb."""
    pdb = mda.Universe(pdb_file)
    assert (pdb.atoms.names == universe.atoms.names).all(), "atoms differ from the build"
    universe.add_TopologyAttr("chainIDs", pdb.atoms.chainIDs)
    if not hasattr(universe.atoms, "elements"):
        universe.add_TopologyAttr("elements", pdb.atoms.elements)
    segids = pdb.residues.segids
    for segid in dict.fromkeys(segids):
        universe.residues[segids == segid].segments = universe.add_Segment(segid=segid)


if condition.config.engine == "gromacs":
    gromacs = run_dir / "gromacs"
    (top,) = gromacs.glob("*.top")
    by_hand = mda.Universe(
        top, gromacs / "prod_centered.xtc", topology_format="ITP", infer_system=True
    )
else:

    def segment_number(path):
        return int(re.search(r"production_(\d+)_trajectory", path.name).group(1))

    segments = sorted(run_dir.glob("production_*/production_*_trajectory.dcd"), key=segment_number)
    topology = run_dir / "system.prmtop"
    if not topology.is_file():  # a run with a restraint, or an older run
        topology = run_dir / "solvated_system.pdb"
    by_hand = mda.Universe(topology, segments)
add_build_ids(by_hand, run_dir / "solvated_system.pdb")

print(by_hand)
print(f"{by_hand.trajectory.n_frames} frames, from {by_hand.trajectory[0].time:.1f} "
      f"to {by_hand.trajectory[-1].time:.1f} ps")
print("Same atoms as PolyzyMD's universe:", (by_hand.atoms.names == u.atoms.names).all())
print("Same first frame:", np.allclose(by_hand.trajectory[0].positions, u.trajectory[0].positions, atol=1e-3))

## 6. Use it on every replicate

When a measurement works here, let PolyzyMD run it on every replicate of every
condition, store it and compare the conditions:

- In Python: `study.timeseries(measure, pz.select("protein"), unit="A")`,
  then `.reduce("mean")` (the docs page "Measure your own quantity on every
  replicate").
- From the project: put the function in `analyses/` and list it under
  `analyses:` in `project.yaml` or `study.yaml` as `function:
  analyses/<file>.py:<function>`, then run `polyzymd analyze --project .`.